# Window Functions and Field Algebra

This notebook sits between `convols.ipynb` and the measurement notebooks. After a catalog has been converted into a `ConvolsData` object, PyHermes lets you manipulate two objects directly:

- `ConvolsData`: a multiresolution field, stored in `epsilon`.
- `WindowFunc`: a Fourier-space filter, stored as `w_kernel` after it is built.

The goal here is to show the arithmetic supported by both objects, then introduce ordinary smoothing windows. These are not 2PCF `pair_window` definitions; pair windows are discussed in `corr2pcf.ipynb`.

In [1]:
from pathlib import Path
import copy
import os

import numpy as np
from numba import njit

from pyhermes.io import ConvolsData, WindowFunc
from pyhermes.utils.window_functions import (
    window_function_cylshell_numba,
    window_function_disk_numba,
    window_function_sphere_numba,
)

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)
elif (cwd / "examples").is_dir():
    os.chdir(cwd / "examples")

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/xutengpeng/xutp/PycharmProjects/PyHermes/examples


## 1. Load Example Fields

`convols.ipynb` builds the files used below. Here we read a data field and a random field. The scalar `rho = 1 / D.V` is the uniform-random shortcut used by later notebooks.

In [2]:
D = ConvolsData(data_path="./output/quijote8000_snap004_sfc.pkl", threads=8)
R = ConvolsData(data_path="./output/random_sfc.pkl", threads=8)

rho = 1.0 / D.V
print(f"D.epsilon shape = {D.epsilon.shape}")
print(f"uniform density rho = {rho:.6e}")

14:19:30 - INFO - pyhermes.io.base:ConvolsData - Reading Convols data from ---> ./output/quijote8000_snap004_sfc.pkl <---
14:19:30 - INFO - pyhermes.io.base:ConvolsData - epsilon: Shape(256, 256, 256), Min = -3.215e-06, Max = 1.343e-05, Mean = 5.96e-08, Sum = 1
14:19:30 - INFO - pyhermes.io.base:ConvolsData - Reading Convols data from ---> ./output/random_sfc.pkl <---
14:19:30 - INFO - pyhermes.io.base:ConvolsData - epsilon: Shape(256, 256, 256), Min = -2.192e-07, Max = 9.48e-07, Mean = 5.96e-08, Sum = 1
D.epsilon shape = (256, 256, 256)
uniform density rho = 5.960464e-08


## 2. `ConvolsData` Arithmetic

`ConvolsData` arithmetic acts on `epsilon` and returns a new `ConvolsData` object. This is useful for building density contrasts or simple derived fields before applying a window.

In [3]:
delta_from_random = D - R
delta_from_uniform = D - rho
mean_field = (D + R) / 2.0
scaled_delta = 2.0 * delta_from_uniform
field_product = D * R

print(delta_from_random.epsilon.shape)
print(mean_field.epsilon.shape)
print(scaled_delta.epsilon.shape)
print(field_product.epsilon.shape)

(256, 256, 256)
(256, 256, 256)
(256, 256, 256)
(256, 256, 256)


Supported field operations are:

```python
D1 + D2
D1 - D2
D1 * D2
D + a
D - a
a - D
a * D
D * a
D / a
```

Here `a` is a scalar. Division is only defined as `D / a`; `a / D` and `D1 / D2` are intentionally left out because their numerical and physical meaning is less clean.

## 3. Basic Smoothing with `WindowFunc`

A `WindowFunc` is built from a small dictionary and the `convols_info` of the field it will filter. The actual kernel is built lazily when the window is used by `D @ W` or when `W.as_array()` is called.

In [4]:
def make_window(params, base=D):
    return WindowFunc(copy.deepcopy(params), base.convols_info, threads=8)

W_sphere20 = make_window({"type": "sphere", "len_args": {"R": 20.0}})
D_sphere20 = D @ W_sphere20
delta_sphere20 = delta_from_uniform @ W_sphere20

print(D_sphere20.epsilon.shape)
print(delta_sphere20.epsilon.shape)

(256, 256, 256)
(256, 256, 256)


## 4. `WindowFunc` Arithmetic

`WindowFunc` arithmetic acts on `w_kernel`. A composite window is materialized: it stores the combined kernel directly instead of trying to reconstruct a new Numba function.

In [5]:
W_gauss8 = make_window({"type": "gaussian", "len_args": {"R": 8.0}})
W_mix = 0.7 * W_sphere20 + 0.3 * W_gauss8
D_mix = D @ W_mix

print(W_mix.type)
print(W_mix.as_array().shape)
print(D_mix.epsilon.shape)

composite
(256, 256, 129)
(256, 256, 256)


Supported window operations are:

```python
W1 + W2
W1 - W2
a * W
W * a
W / a
-W
```

Both windows in `W1 + W2` or `W1 - W2` must be built for the same grid: the same `J`, `box_size`, `phi_resolution`, `wavelet_mode`, `wavelet_level`, `bandwidth`, and kernel shape.

## 5. Built-In Ordinary Windows

The dictionaries below are ordinary smoothing/filter windows. They can be used as `window`, `window1`, `window2`, or a direct `WindowFunc` input. They are not yet 2PCF `pair_window` settings.

In [6]:
builtin_window_params = {
    "sphere": {
        "type": "sphere",
        "len_args": {"R": 20.0},
    },
    "gaussian": {
        "type": "gaussian",
        "len_args": {"R": 8.0},
    },
    "shell": {
        "type": "shell",
        "len_args": {"R": 20.0},
    },
    "cubic": {
        "type": "cubic",
        "len_args": {"Lx": 20.0, "Ly": 20.0, "Lz": 20.0},
    },
    "ring": {
        "type": "ring",
        "len_args": {"R": 20.0, "H": 10.0},
        "los_args": [0.0, 0.0, 1.0],
    },
    "disk": {
        "type": "disk",
        "len_args": {"R": 20.0, "H": 10.0},
        "los_args": [0.0, 0.0, 1.0],
    },
    "cylshell": {
        "type": "cylshell",
        "len_args": {"R": 20.0, "H": 10.0},
        "los_args": [0.0, 0.0, 1.0],
    },
    "cylinder": {
        "type": "cylinder",
        "len_args": {"R": 20.0, "H": 10.0},
        "los_args": [0.0, 0.0, 1.0],
    },
}

for name, params in builtin_window_params.items():
    print(f"{name:9s}: {params}")

sphere   : {'type': 'sphere', 'len_args': {'R': 20.0}}
gaussian : {'type': 'gaussian', 'len_args': {'R': 8.0}}
shell    : {'type': 'shell', 'len_args': {'R': 20.0}}
cubic    : {'type': 'cubic', 'len_args': {'Lx': 20.0, 'Ly': 20.0, 'Lz': 20.0}}
ring     : {'type': 'ring', 'len_args': {'R': 20.0, 'H': 10.0}, 'los_args': [0.0, 0.0, 1.0]}
disk     : {'type': 'disk', 'len_args': {'R': 20.0, 'H': 10.0}, 'los_args': [0.0, 0.0, 1.0]}
cylshell : {'type': 'cylshell', 'len_args': {'R': 20.0, 'H': 10.0}, 'los_args': [0.0, 0.0, 1.0]}
cylinder : {'type': 'cylinder', 'len_args': {'R': 20.0, 'H': 10.0}, 'los_args': [0.0, 0.0, 1.0]}


In [7]:
W_cubic20 = make_window(builtin_window_params["cubic"])
D_cubic20 = D @ W_cubic20
print(D_cubic20.epsilon.shape)

(256, 256, 256)


The line-of-sight windows (`ring`, `disk`, `cylshell`, `cylinder`) use `los_args`. Here `H` is a distance along the line of sight; for `cylshell` and `cylinder`, `H` is the half-height of the finite cylinder.

## 6. A Minimal Custom Window

A custom window is a Numba function for the Fourier-space kernel. The first three arguments must be `ki`, `kj`, and `kk`. Length parameters go in `len_args`; dimensionless controls go in `other_args`; line-of-sight components go in `los_args`.

In [8]:
@njit
def window_function_cos_shell_numba(ki, kj, kk, R, amplitude=1.0):
    k = np.sqrt(ki * ki + kj * kj + kk * kk)
    q = 2.0 * np.pi * k * R
    return amplitude * np.cos(q)

cos_shell_params = {
    "type": "cos_shell",
    "func": window_function_cos_shell_numba,
    "len_args": {"R": 20.0},
    "other_args": {"amplitude": 1.0},
    "kernel_mode": "octant",
}

W_cos_shell = make_window(cos_shell_params)
D_cos_shell = D @ W_cos_shell
print(D_cos_shell.epsilon.shape)

(256, 256, 256)


`kernel_mode="octant"` is fast and works when the kernel is invariant under independent sign flips of `kx`, `ky`, and `kz`. For a general custom window, use `kernel_mode="full_rfft"`; for line-of-sight windows you can often use `kernel_mode="auto"`.

## 7. Compose Windows from Built-In Pieces

Composite windows are often easier to build from existing windows than to write from scratch. The next examples show both routes and verify that the kernels agree.

### 7.1 Finite-Thickness Spherical Shell

A finite shell between `R_in` and `R_out` is a volume-weighted difference of two spherical top-hats:

```python
W = (R_out**3 * W_sphere(R_out) - R_in**3 * W_sphere(R_in)) / (R_out**3 - R_in**3)
```

In [9]:
@njit
def window_function_thick_shell_numba(ki, kj, kk, R_in, R_out):
    V_in = R_in**3
    V_out = R_out**3
    denom = V_out - V_in
    if denom == 0.0:
        k = np.sqrt(ki**2 + kj**2 + kk**2)
        q_out = 2.0 * np.pi * k * R_out
        if q_out == 0.0:
            return 1.0
        return np.sin(q_out) / q_out

    W_in = window_function_sphere_numba(ki, kj, kk, R_in)
    W_out = window_function_sphere_numba(ki, kj, kk, R_out)
    return (W_out * V_out - W_in * V_in) / denom

In [10]:
R_in, R_out = 10.0, 20.0
W_inner = make_window({"type": "sphere", "len_args": {"R": R_in}})
W_outer = make_window({"type": "sphere", "len_args": {"R": R_out}})

W_thick_from_builtin = (W_outer * R_out**3 - W_inner * R_in**3) / (R_out**3 - R_in**3)

W_thick_custom = make_window({
    "type": "thick_shell",
    "func": window_function_thick_shell_numba,
    "len_args": {"R_in": R_in, "R_out": R_out},
    "kernel_mode": "octant",
})

np.allclose(W_thick_custom.as_array(), W_thick_from_builtin.as_array())

True

### 7.2 Cylindrical Surface

A cylindrical surface can be composed from the side surface (`cylshell`) and the two caps (`disk`). The weights are proportional to their areas, giving

```python
W = (2H * W_cylshell + R * W_disk) / (2H + R)
```

In [11]:
@njit
def window_function_cylsurf_numba(ki, kj, kk, R, H, nx=0.0, ny=0.0, nz=1.0):
    denom = 2.0 * H + R
    if denom == 0.0:
        return 1.0
    win_cylshell = window_function_cylshell_numba(ki, kj, kk, R, H, nx, ny, nz)
    win_disk = window_function_disk_numba(ki, kj, kk, R, H, nx, ny, nz)
    return (2.0 * H * win_cylshell + R * win_disk) / denom

In [12]:
R_cyl, H_cyl = 10.0, 20.0
los_z = [0.0, 0.0, 1.0]

W_cylshell = make_window({
    "type": "cylshell",
    "len_args": {"R": R_cyl, "H": H_cyl},
    "los_args": los_z,
})
W_disk = make_window({
    "type": "disk",
    "len_args": {"R": R_cyl, "H": H_cyl},
    "los_args": los_z,
})

W_cylsurf_from_builtin = (2.0 * H_cyl * W_cylshell + R_cyl * W_disk) / (2.0 * H_cyl + R_cyl)

W_cylsurf_custom = make_window({
    "type": "cylsurf",
    "func": window_function_cylsurf_numba,
    "len_args": {"R": R_cyl, "H": H_cyl},
    "los_args": los_z,
    "kernel_mode": "auto",
})

np.allclose(W_cylsurf_custom.as_array(), W_cylsurf_from_builtin.as_array())

True

## 8. Smoothing Windows versus Pair Windows

Everything above treats `WindowFunc` as an ordinary smoothing/filter window: it acts on a field through `D @ W`. PyHermes also uses window functions as 2PCF `pair_window` definitions, where the window selects a separation bin instead of smoothing a field. That role is central to `Corr_2PCF` and is introduced in `corr2pcf.ipynb`.